### Spearman analyses (78 STG samples)
Association of per-sample coding-variant burden with nuclei number and post-mortem interval (PMI).

In [ ]:

import numpy as np, pandas as pd, statsmodels.api as sm
from scipy import stats

## Data imported from supplementary tables only for the STG's
raw = """id nuclei n_var pmi
A1_C 187 356 48:00 A2_C 34 77 07:04 A3_P 599 1089 27:00 A4_P 818 1838 04:45 A5_P 111 235 12:00
B1_C 367 625 05:30 B2_C 48 73 52:00 B3_P 1137 2623 07:30 B4_P 49 96 08:45 B5_P 1090 2039 49:00
C1_C 766 1584 06:00 C3_P 2404 4077 08:45 C4_P 831 1805 47:00 C5_P 893 1839 07:42 D1_C 135 423 72:00
D2_C 134 463 66:00 D3_P 681 1960 05:00 D4_P 267 800 11:00 D5_P 1231 3339 05:00 E1_C 161 270 71:00
E2_C 312 610 07:30 E3_P 129 177 21:00 E4_P 4296 6338 09:00 E5_P 674 1110 30:00 F1_C 98 288 18:00
F2_C 211 600 31:00 F3_P 385 1177 02:00 F4_P 748 2095 17:00 F5_P 573 1654 58:00 G1_C 52 117 39:00
G3_P 34 77 42:00 G4_P 1693 3337 05:00 G5_P 2085 3714 30:00 H2_C 54 86 NA H3_P 93 136 24:00
H4_P 157 259 06:00 H5_P 101 162 07:00 I1_C 111 198 16:00 I2_C 93 197 NA I3_P 1916 2891 24:00
I4_P 80 179 22:00 I5_P 385 716 13:00 J1_C 568 888 08:00 J2_C 39 50 32:00 J3_P 1460 2856 12:00
J4_P 54 119 05:00 J5_P 527 854 02:30 K1_C 32 58 54:00 K3_P 109 237 04:00 K5_P 55 71 45:00
L2_C 44 77 NA L3_P 40 75 14:00 L5_P 82 149 55:00 M2_P 113 252 NA M3_P 242 487 10:30
M5_C 27 73 NA N2_P 810 2459 06:00 N3_P 191 570 24:45 N4_P 883 2652 04:35
N5_P 73 225 03:00 O1_P 3463 4987 11:30 P1_P 47 94 04:00 P2_C 50 106 05:00 P4_P 75 160 NA
P5_C 27 48 05:45 Q1_C 1333 2070 07:35 Q2_C 90 167 12:00 Q3_P 765 1396 24:00
R1_P 37 67 NA R2_C 23 37 04:30 R3_C 66 113 NA R4_P 123 197 08:30
S4_P 94 147 10:00 S5_P 181 322 23:00 T2_P 115 169 24:00 T3_P 401 590 08:00 T4_P 1018 1874 NA
T5_P 320 532 NA"""

tok = raw.split()

In [3]:
df = pd.DataFrame(np.array(tok[4:]).reshape(-1, 4), columns=tok[:4])
df[["nuclei", "n_var"]] = df[["nuclei", "n_var"]].astype(int)
df["pmi_h"] = df.pmi.replace("NA", np.nan).dropna().str.split(":").apply(lambda x: int(x[0]) + int(x[1]) / 60)
df["group"] = np.where(df.id.str.endswith("_P"), "patient", "control")
print(f"samples: {len(df)} ({(df.group=='patient').sum()} patients, {(df.group=='control').sum()} controls); with PMI: {df.pmi_h.notna().sum()}")

# 1. Burden vs nuclei
r = stats.spearmanr(df.nuclei, df.n_var); print(f"\nSpearman nuclei vs variants: rho = {r.statistic:.3f}, P = {r.pvalue:.2e}")

d = df.dropna(subset=["pmi_h"]).copy()
# 2. Crude and nuclei-adjusted (partial) rank correlation with PMI
r0 = stats.spearmanr(d.pmi_h, d.n_var)
rk = d[["n_var", "pmi_h", "nuclei"]].rank()
res = lambda y: sm.OLS(rk[y], sm.add_constant(rk.nuclei)).fit().resid          # residualise ranks on nuclei
rp = stats.pearsonr(res("n_var"), res("pmi_h"))
print(f"Spearman PMI vs variants (crude): rho = {r0.statistic:.3f}, P = {r0.pvalue:.3f}  (n = {len(d)})")
print(f"Partial Spearman, adjusted for nuclei: rho = {rp.statistic:.3f}, P = {rp.pvalue:.3f}")


# Sensitivity: PMI vs variants per nucleus
r2 = stats.spearmanr(d.pmi_h, d.n_var / d.nuclei)
print(f"Spearman PMI vs variants per nucleus: rho = {r2.statistic:.3f}, P = {r2.pvalue:.3f}")

samples: 78 (52 patients, 26 controls); with PMI: 68

Spearman nuclei vs variants: rho = 0.982, P = 7.68e-57
Spearman PMI vs variants (crude): rho = -0.134, P = 0.277  (n = 68)
Partial Spearman, adjusted for nuclei: rho = -0.219, P = 0.073
Spearman PMI vs variants per nucleus: rho = -0.130, P = 0.292
